In [3]:
import pandas as pd

salary_df = pd.read_csv("/Users/savithrallapalli/Documents/NBA-Risk-Analysis/data/raw/salaries/nba_salaries.csv")

print(salary_df.head())
print(salary_df.columns.tolist())
print(salary_df.shape)

   Unnamed: 0        Player Name    Salary Position  Age     Team  GP  GS  \
0           0      Stephen Curry  48070014       PG   34      GSW  56  56   
1           1          John Wall  47345760       PG   32      LAC  34   3   
2           2  Russell Westbrook  47080179       PG   34  LAL/LAC  73  24   
3           3       LeBron James  44474988       PF   38      LAL  55  54   
4           4       Kevin Durant  44119845       PF   34  BRK/PHO  47  47   

     MP    FG  ...  ORB  DRB  TRB  AST  STL  BLK  TOV   PF   PTS  \
0  34.7  10.0  ...  0.7  5.4  6.1  6.3  0.9  0.4  3.2  2.1  29.4   
1  22.2   4.1  ...  0.4  2.3  2.7  5.2  0.8  0.4  2.4  1.7  11.4   
2  29.1   5.9  ...  1.2  4.6  5.8  7.5  1.0  0.5  3.5  2.2  15.9   
3  35.5  11.1  ...  1.2  7.1  8.3  6.8  0.9  0.6  3.2  1.6  28.9   
4  35.6  10.3  ...  0.4  6.3  6.7  5.0  0.7  1.4  3.3  2.1  29.1   

   Player-additional  
0          curryst01  
1           walljo01  
2          westbru01  
3          jamesle01  
4          du

In [4]:
stats_df = pd.read_csv("/Users/savithrallapalli/Documents/NBA-Risk-Analysis/data/raw/statistics/player_stats_2022.csv")

print(stats_df.shape)
stats_df.head()

(539, 67)


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612742,DAL,22.0,15,5,10,0.333,...,358,475,457,429,462,482,253,39,466,2
1,1631260,AJ Green,AJ,1610612749,MIL,23.0,35,27,8,0.771,...,498,454,518,392,309,455,253,39,436,1
2,1631100,AJ Griffin,AJ,1610612737,ATL,19.0,72,34,38,0.472,...,268,394,422,217,169,287,253,39,273,1
3,203932,Aaron Gordon,Aaron,1610612743,DEN,27.0,68,45,23,0.662,...,32,205,58,82,4,77,56,39,89,1
4,1628988,Aaron Holiday,Aaron,1610612737,ATL,26.0,63,32,31,0.508,...,315,377,356,415,212,413,253,39,416,1


In [5]:
stats_df[["PLAYER_NAME"]].head()

,PLAYER_NAME
0,A.J. Lawson
1,AJ Green
2,AJ Griffin
3,Aaron Gordon
4,Aaron Holiday


In [6]:
salary_df[["Player Name"]].head()

,Player Name
0,Stephen Curry
1,John Wall
2,Russell Westbrook
3,LeBron James
4,Kevin Durant


In [7]:
salary_df = salary_df.rename(
    columns={
        "Player Name": "PLAYER_NAME",
        "Salary": "SALARY"
    }
)

merged_df = stats_df.merge(
    salary_df[["PLAYER_NAME", "SALARY"]],
    on="PLAYER_NAME",
    how="inner"
)

print(merged_df.shape)
merged_df[["PLAYER_NAME", "SALARY"]].sample(10)

(458, 68)


,PLAYER_NAME,SALARY
374,Raul Neto,2463490
450,Xavier Moon,116986
424,Torrey Craig,5121951
153,Grayson Allen,8925000
210,Jaylen Brown,29776785
421,Thomas Bryant,1968175
176,Jaden Hardy,1017781
346,Nickeil Alexander-Walker,5009633
394,Scottie Barnes,7644600
252,Justin Holiday,5987975


In [8]:
merged_df.to_csv("/Users/savithrallapalli/Documents/NBA-Risk-Analysis/data/merged_nba_salary_stats_2022.csv", index=False)

In [8]:
merged_df["performance_score"] = (
    merged_df["PTS"]
    + 1.2 * merged_df["REB"]
    + 1.5 * merged_df["AST"]
    + 3 * merged_df["STL"]
    + 3 * merged_df["BLK"]
    - 1.5 * merged_df["TOV"]
)

merged_df["availability"] = merged_df["GP"] / 82

merged_df["risk_adj_performance"] = (
    merged_df["performance_score"]
    * merged_df["availability"]
)

merged_df["value_per_million"] = (
    merged_df["risk_adj_performance"]
    / (merged_df["SALARY"] / 1_000_000)
)
merged_df[
    [
        "PLAYER_NAME",
        "SALARY",
        "risk_adj_performance",
        "value_per_million"
    ]
].sort_values(
    "value_per_million",
    ascending=False
).head(20)


,PLAYER_NAME,SALARY,risk_adj_performance,value_per_million
373,RaiQuan Gray,5849,0.418293,71.515248
175,Jacob Gilyard,5849,0.296341,50.665321
402,Skylar Mays,116574,2.388293,20.487353
254,Justin Minaya,35096,0.695610,19.820200
141,Gabe York,32171,0.550610,17.115096
22,Anthony Lamb,694878,10.736585,15.451037
426,Tre Jones,1782621,23.982439,13.453470
400,Shaquille Harrison,134862,1.773171,13.148038
108,Desmond Bane,2130240,24.756098,11.621272
157,Immanuel Quickley,2316240,26.512683,11.446432


In [9]:
analysis_df = merged_df[
    (merged_df["GP"] >= 40) &
    (merged_df["SALARY"] >= 1_500_000)
].copy()

analysis_df[
    [
        "PLAYER_NAME",
        "SALARY",
        "GP",
        "risk_adj_performance",
        "value_per_million"
    ]
].sort_values(
    "value_per_million",
    ascending=False
).head(20)

,PLAYER_NAME,SALARY,GP,risk_adj_performance,value_per_million
426,Tre Jones,1782621,68,23.982439,13.453470
108,Desmond Bane,2130240,58,24.756098,11.621272
157,Immanuel Quickley,2316240,81,26.512683,11.446432
23,Austin Reaves,1563518,64,17.053659,10.907235
25,Ayo Dosunmu,1563518,80,16.936585,10.832357
178,Jaden McDaniels,2161440,79,22.380122,10.354265
321,Max Strus,1815677,80,18.770732,10.338145
84,Daniel Gafford,1930681,78,19.804390,10.257723
188,Jalen McDaniels,1930681,80,18.887805,9.782975
339,Naji Marshall,1782621,77,16.827317,9.439649


In [10]:
from sklearn.linear_model import LinearRegression
import numpy as np
X = analysis_df[["SALARY"]]
y = analysis_df["risk_adj_performance"]
model = LinearRegression()

model.fit(X, y)

analysis_df["expected_performance"] = model.predict(X)

analysis_df["alpha"] = (
    analysis_df["risk_adj_performance"]
    - analysis_df["expected_performance"]
)

analysis_df[
    [
        "PLAYER_NAME",
        "SALARY",
        "risk_adj_performance",
        "expected_performance",
        "alpha"
    ]
].sort_values(
    "alpha",
    ascending=False
).head(20)

,PLAYER_NAME,SALARY,risk_adj_performance,expected_performance,alpha
20,Anthony Edwards,10733400,38.642561,17.379138,21.263423
114,Domantas Sabonis,21100000,42.737073,22.947399,19.789674
132,Evan Mobley,8478720,34.104878,16.168071,17.936807
241,Josh Giddey,6287400,32.235122,14.991037,17.244085
213,Jayson Tatum,30351780,43.867561,27.916852,15.950709
437,Tyrese Haliburton,4215120,29.359024,13.877943,15.481081
394,Scottie Barnes,7644600,31.100488,15.720036,15.380452
237,Jordan Poole,3901399,29.040000,13.709433,15.330567
170,Ja Morant,12119440,33.125976,18.123628,15.002348
250,Julius Randle,23760000,39.204268,24.376178,14.828091


In [11]:
analysis_df[
    [
        "PLAYER_NAME",
        "SALARY",
        "risk_adj_performance",
        "expected_performance",
        "alpha"
    ]
].sort_values(
    "alpha",
    ascending=True
).head(20)

,PLAYER_NAME,SALARY,risk_adj_performance,expected_performance,alpha
27,Ben Simmons,35448672,13.245366,30.654570,-17.409204
124,Duncan Robinson,16902000,5.029756,20.692507,-15.662751
376,Richaun Holmes,11215260,3.293415,17.637962,-14.344547
275,Kevin Love,30556968,14.048293,28.027065,-13.978773
35,Bradley Beal,43279250,22.213415,34.860645,-12.647231
150,Gordon Hayward,30075000,15.859756,27.768184,-11.908428
448,Will Barton,15059712,9.171707,19.702950,-10.531243
172,JaVale McGee,5461219,4.404878,14.547266,-10.142388
92,Davon Reed,1902133,2.611463,12.635557,-10.024094
286,Kyle Lowry,28333334,16.848780,26.832674,-9.983894


In [12]:
analysis_df.to_csv(
    "/Users/savithrallapalli/Documents/NBA-Risk-Analysis/data/nba_contract_valuation_2022.csv",
    index=False
)